In [27]:
import joblib
import pandas
import numpy
import time

from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, StackingRegressor
from sklearn.preprocessing import StandardScaler, Normalizer, normalize
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from statsmodels.tsa.arima.model import ARIMA

from skforecast.direct import ForecasterDirect

In [16]:
df = pandas.read_csv('WalesData/WalesPops-Transposed.csv')
df['Date'] = pandas.to_datetime(df['Date'], format='%Y')
df.set_index('Date', inplace=True)
df = df.asfreq('YS')
df

,Blaenau Gwent,Bridgend,Caerphilly,Cardiff,Carmarthenshire,Ceredigion,Conwy,Denbighshire,Flintshire,Gwynedd,...,Monmouthshire,Neath Port Talbot,Newport,Pembrokeshire,Powys,Rhondda Cynon Taf,Swansea,Torfaen,Vale of Glamorgan,Wrexham
Date,,,,,,,,,,,,,,,,,,,,,
1991-01-01,72666,129477,170615,296941,169725,65933,107951,89395,142036,115007,...,80209,138844,135479,112446,119703,234917,229743,90961,118053,124180
1992-01-01,72592,129796,170129,297429,169265,67536,107782,89119,143493,115426,...,80207,138496,135515,113146,120368,235202,229610,91188,117605,124627
1993-01-01,72521,130129,170243,299185,169255,68854,108142,89375,144049,115717,...,81637,138219,135271,112885,121072,235468,229349,90899,117050,124949
1994-01-01,72770,130063,169559,299085,169962,69937,108891,89572,144539,116330,...,83200,138597,135461,113086,121400,235147,228239,90928,116850,125298
1995-01-01,72569,129466,168186,300764,171060,70818,108940,89911,144844,117526,...,84315,137751,135140,112249,121843,235553,227765,90955,116347,125431
1996-01-01,72289,128667,167385,305345,171145,70609,107897,90623,144106,117486,...,85209,137327,134707,112141,123954,235369,227199,91268,116327,125527
1997-01-01,72031,127996,167532,307321,171534,71807,108174,90616,144643,117606,...,85209,136679,135241,111640,124604,236043,226764,91364,116950,125637
1998-01-01,71292,128652,167921,308601,172039,72518,108470,90951,146261,117384,...,83921,136085,137328,111814,125438,234474,225702,91355,118297,126178
1999-01-01,71028,128585,168566,310320,172329,74046,108322,91509,146824,116362,...,83980,135356,136615,111667,125632,233904,225080,91129,117944,126907


In [3]:
ensemble = joblib.load('stackensemble')
forest = joblib.load('randforest')
ensemble

,estimators,"[('mlp', ...), ('knn', ...)]"
,final_estimator,LinearRegression()
,cv,None
,n_jobs,2
,passthrough,True
,verbose,0
,loss,'squared_error'
,hidden_layer_sizes,50
,activation,'tanh'
,solver,'lbfgs'
,alpha,0.0001


In [22]:
forecaster1 = ForecasterDirect(
    estimator=ensemble,
    lags=1,
    steps=10,
    differentiation=1
)

forecaster2 = ForecasterDirect(
    estimator=forest,
    lags=1,
    differentiation=1,
    steps=10
)

forecasters = [forecaster1, forecaster2]

for caster in forecasters:
    caster.fit(y=df['Blaenau Gwent'])

# ens_results = forecaster1.predict(df['Blaenau Gwent'].values.reshape(-1,1))

# for_results = forecaster2.predict(df['Blaenau Gwent'].values.reshape(-1,1))

ens_results, for_results = (caster.predict() for caster in forecasters)

/home/jl/Documents/spfxvenv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:602: ConvergenceWarning: lbfgs failed to converge after 1500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/home/jl/Documents/spfxvenv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:602: ConvergenceWarning: lbfgs failed to converge after 1500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_it

╭─────────────────────────────── IgnoredArgumentWarning ───────────────────────────────╮
│ The number of bins has been reduced from 10 to 1 due to duplicated edges caused by   │
│ repeated predicted values.                                                           │
│                                                                                      │
│ Category : skforecast.exceptions.IgnoredArgumentWarning                              │
│ Location :                                                                           │
│ /home/jl/Documents/spfxvenv/lib/python3.12/site-packages/skforecast/preprocessing/pr │
│ eprocessing.py:2515                                                                  │
│ Suppress : warnings.simplefilter('ignore', category=IgnoredArgumentWarning)          │
╰──────────────────────────────────────────────────────────────────────────────────────╯

In [23]:
ens_results

2024-01-01    67474.158215
2025-01-01    67541.792829
2026-01-01    67383.351274
2027-01-01    67066.062497
2028-01-01    66757.042317
2029-01-01    66080.868388
2030-01-01    65688.200901
2031-01-01    65111.685965
2032-01-01    64735.245893
2033-01-01    64506.118842
Freq: YS-JAN, Name: pred, dtype: float64

In [24]:
for_results

2024-01-01    121333.0
2025-01-01    175310.0
2026-01-01    229287.0
2027-01-01    283264.0
2028-01-01    337241.0
2029-01-01    391218.0
2030-01-01    445195.0
2031-01-01    499172.0
2032-01-01    553149.0
2033-01-01    607126.0
Freq: YS-JAN, Name: pred, dtype: float64

In [28]:
def time_forecaster(forecaster):
    start = time.perf_counter()
    
    for column in df.columns[0:5]:
        forecaster.fit(df[column])
        forecaster.predict()

    end = time.perf_counter()

    return end - start
    

print(f'''
stackingensemble: {time_forecaster(forecaster1)}
randomforest: {time_forecaster(forecaster2)}
''')

/home/jl/Documents/spfxvenv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:602: ConvergenceWarning: lbfgs failed to converge after 1500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/home/jl/Documents/spfxvenv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:602: ConvergenceWarning: lbfgs failed to converge after 1500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_it

╭─────────────────────────────── IgnoredArgumentWarning ───────────────────────────────╮
│ The number of bins has been reduced from 10 to 1 due to duplicated edges caused by   │
│ repeated predicted values.                                                           │
│                                                                                      │
│ Category : skforecast.exceptions.IgnoredArgumentWarning                              │
│ Location :                                                                           │
│ /home/jl/Documents/spfxvenv/lib/python3.12/site-packages/skforecast/preprocessing/pr │
│ eprocessing.py:2515                                                                  │
│ Suppress : warnings.simplefilter('ignore', category=IgnoredArgumentWarning)          │
╰──────────────────────────────────────────────────────────────────────────────────────╯

/home/jl/Documents/spfxvenv/lib/python3.12/site-packages/sklearn/ensemble/_forest.py:465: UserWarning: Warm-start fitting without increasing n_estimators does not fit new trees.
  warn(
/home/jl/Documents/spfxvenv/lib/python3.12/site-packages/sklearn/ensemble/_forest.py:465: UserWarning: Warm-start fitting without increasing n_estimators does not fit new trees.
  warn(
/home/jl/Documents/spfxvenv/lib/python3.12/site-packages/sklearn/ensemble/_forest.py:465: UserWarning: Warm-start fitting without increasing n_estimators does not fit new trees.
  warn(
/home/jl/Documents/spfxvenv/lib/python3.12/site-packages/sklearn/ensemble/_forest.py:465: UserWarning: Warm-start fitting without increasing n_estimators does not fit new trees.
  warn(
/home/jl/Documents/spfxvenv/lib/python3.12/site-packages/sklearn/ensemble/_forest.py:465: UserWarning: Warm-start fitting without increasing n_estimators does not fit new trees.
  warn(
/home/jl/Documents/spfxvenv/lib/python3.12/site-packages/sklearn/ensem

╭─────────────────────────────── IgnoredArgumentWarning ───────────────────────────────╮
│ The number of bins has been reduced from 10 to 1 due to duplicated edges caused by   │
│ repeated predicted values.                                                           │
│                                                                                      │
│ Category : skforecast.exceptions.IgnoredArgumentWarning                              │
│ Location :                                                                           │
│ /home/jl/Documents/spfxvenv/lib/python3.12/site-packages/skforecast/preprocessing/pr │
│ eprocessing.py:2515                                                                  │
│ Suppress : warnings.simplefilter('ignore', category=IgnoredArgumentWarning)          │
╰──────────────────────────────────────────────────────────────────────────────────────╯

/home/jl/Documents/spfxvenv/lib/python3.12/site-packages/sklearn/ensemble/_forest.py:465: UserWarning: Warm-start fitting without increasing n_estimators does not fit new trees.
  warn(
/home/jl/Documents/spfxvenv/lib/python3.12/site-packages/sklearn/ensemble/_forest.py:465: UserWarning: Warm-start fitting without increasing n_estimators does not fit new trees.
  warn(
/home/jl/Documents/spfxvenv/lib/python3.12/site-packages/sklearn/ensemble/_forest.py:465: UserWarning: Warm-start fitting without increasing n_estimators does not fit new trees.
  warn(
/home/jl/Documents/spfxvenv/lib/python3.12/site-packages/sklearn/ensemble/_forest.py:465: UserWarning: Warm-start fitting without increasing n_estimators does not fit new trees.
  warn(
/home/jl/Documents/spfxvenv/lib/python3.12/site-packages/sklearn/ensemble/_forest.py:465: UserWarning: Warm-start fitting without increasing n_estimators does not fit new trees.
  warn(
/home/jl/Documents/spfxvenv/lib/python3.12/site-packages/sklearn/ensem

...

/home/jl/Documents/spfxvenv/lib/python3.12/site-packages/sklearn/ensemble/_forest.py:465: UserWarning: Warm-start fitting without increasing n_estimators does not fit new trees.
  warn(
/home/jl/Documents/spfxvenv/lib/python3.12/site-packages/sklearn/ensemble/_forest.py:465: UserWarning: Warm-start fitting without increasing n_estimators does not fit new trees.
  warn(
/home/jl/Documents/spfxvenv/lib/python3.12/site-packages/sklearn/ensemble/_forest.py:465: UserWarning: Warm-start fitting without increasing n_estimators does not fit new trees.
  warn(
/home/jl/Documents/spfxvenv/lib/python3.12/site-packages/sklearn/ensemble/_forest.py:465: UserWarning: Warm-start fitting without increasing n_estimators does not fit new trees.
  warn(
/home/jl/Documents/spfxvenv/lib/python3.12/site-packages/sklearn/ensemble/_forest.py:465: UserWarning: Warm-start fitting without increasing n_estimators does not fit new trees.
  warn(
/home/jl/Documents/spfxvenv/lib/python3.12/site-packages/sklearn/ensem

╭─────────────────────────────── IgnoredArgumentWarning ───────────────────────────────╮
│ The number of bins has been reduced from 10 to 1 due to duplicated edges caused by   │
│ repeated predicted values.                                                           │
│                                                                                      │
│ Category : skforecast.exceptions.IgnoredArgumentWarning                              │
│ Location :                                                                           │
│ /home/jl/Documents/spfxvenv/lib/python3.12/site-packages/skforecast/preprocessing/pr │
│ eprocessing.py:2515                                                                  │
│ Suppress : warnings.simplefilter('ignore', category=IgnoredArgumentWarning)          │
╰──────────────────────────────────────────────────────────────────────────────────────╯

/home/jl/Documents/spfxvenv/lib/python3.12/site-packages/sklearn/ensemble/_forest.py:465: UserWarning: Warm-start fitting without increasing n_estimators does not fit new trees.
  warn(
/home/jl/Documents/spfxvenv/lib/python3.12/site-packages/sklearn/ensemble/_forest.py:465: UserWarning: Warm-start fitting without increasing n_estimators does not fit new trees.
  warn(
/home/jl/Documents/spfxvenv/lib/python3.12/site-packages/sklearn/ensemble/_forest.py:465: UserWarning: Warm-start fitting without increasing n_estimators does not fit new trees.
  warn(
/home/jl/Documents/spfxvenv/lib/python3.12/site-packages/sklearn/ensemble/_forest.py:465: UserWarning: Warm-start fitting without increasing n_estimators does not fit new trees.
  warn(
/home/jl/Documents/spfxvenv/lib/python3.12/site-packages/sklearn/ensemble/_forest.py:465: UserWarning: Warm-start fitting without increasing n_estimators does not fit new trees.
  warn(
/home/jl/Documents/spfxvenv/lib/python3.12/site-packages/sklearn/ensem

╭─────────────────────────────── IgnoredArgumentWarning ───────────────────────────────╮
│ The number of bins has been reduced from 10 to 1 due to duplicated edges caused by   │
│ repeated predicted values.                                                           │
│                                                                                      │
│ Category : skforecast.exceptions.IgnoredArgumentWarning                              │
│ Location :                                                                           │
│ /home/jl/Documents/spfxvenv/lib/python3.12/site-packages/skforecast/preprocessing/pr │
│ eprocessing.py:2515                                                                  │
│ Suppress : warnings.simplefilter('ignore', category=IgnoredArgumentWarning)          │
╰──────────────────────────────────────────────────────────────────────────────────────╯


stackingensemble: 25.625094652999906
randomforest: 26.18471162699825

